# Phase 5 — Random Forest evaluation

This notebook fits **Random Forest** on `processed/train.csv` only. The validation split selects the F1 operating threshold. The untouched test split is used once for final metrics.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "processed" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ml.evaluation.pipeline import load_splits, build_model_specs, classification_metrics, select_threshold
splits = load_splits(ROOT)
X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]
X_test, y_test = splits["test"]
models = build_model_specs(X_train, y_train)
MODEL_NAME = 'random_forest'
estimator = models[MODEL_NAME].fit(X_train, y_train)
validation_scores = estimator.predict_proba(X_validation)[:, 1]
selection = select_threshold(y_validation, validation_scores, objective="f1")
test_scores = estimator.predict_proba(X_test)[:, 1]
metrics = pd.DataFrame([
    {"model": MODEL_NAME, "split": "validation", **classification_metrics(y_validation, validation_scores, selection.threshold)},
    {"model": MODEL_NAME, "split": "test", **classification_metrics(y_test, test_scores, selection.threshold)},
])
metrics


## Metrics and interpretation

The table includes ROC-AUC, average precision, **log loss**, Brier score, and thresholded metrics. Log loss evaluates probability quality; lower is better. The visualization uses validation predictions only and is descriptive, not additional tuning.


In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_validation, validation_scores, name="validation", ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_validation, validation_scores, name="validation", ax=axes[1])
axes[0].set_title("Validation ROC curve")
axes[1].set_title("Validation precision-recall curve")
fig.suptitle(MODEL_NAME)
fig.tight_layout()
plt.show()
plt.close(fig)
